# Human Validation Agreement Analysis

This Colab notebook compares:

- 4o-mini holistic 5-class labels against each human annotator's overall sentiment.
- 4o-mini aspect labels against each human annotator's aspect labels.
- Annotator 1 against Annotator 2 over the overlapping completed records.

The notebook analyzes one cumulative workbook per annotator. By default it uses the `*_complete` workbook when available; otherwise it uses the most recently modified annotator workbook in `labeling output/`.


In [ ]:
import os
# If running in Colab, mount Google Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Google Drive mount skipped: {exc}")

from collections import Counter
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    display
except NameError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string(index=False))
        else:
            print(obj)

import os
from pathlib import Path

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents,
                 Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

PROJECT_ROOT = resolve_project_root()

BASE_DIR = PROJECT_ROOT / "00_human_validation"
KEY_PATH = BASE_DIR / "subset" / "human_validation_sample_key_500.csv"
# Frozen annotation release (protocol v2_ef356fdf87): the LLM labels compared with the annotators come from here
RELEASE_PATH = BASE_DIR.parent / "00_labelling" / "output_v2" / "v2_ef356fdf87" / "drugscom_plus_ai_labels_v2_ef356fdf87.csv"
LABEL_DIR = BASE_DIR / "outputs"
OUT_DIR = Path(os.environ.get("HV_OUT_DIR", LABEL_DIR / "agreement_analysis_v2"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("KEY_PATH:", KEY_PATH)
print("LABEL_DIR:", LABEL_DIR)
print("OUT_DIR:", OUT_DIR)

SENT5 = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
ASPECT_VALUES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
ASPECTS = ["efficacy", "safety", "burden", "cost"]
ANNOT_COLS = [
    "human_overall_sentiment",
    "human_efficacy",
    "human_safety",
    "human_burden",
    "human_cost",
    "human_uncertain",
]
SOURCE_COLS = ["sample_id", "source_row_id", "ingredient_norm", "condition_norm", "review"]
YES_NO = ["NO", "YES"]

def canon_series(s):
    return s.astype("string").str.strip().str.upper().replace({"": pd.NA, "NAN": pd.NA})


def norm_ws(x):
    return re.sub(r"\s+", " ", str(x)).strip()


def find_cumulative_annotator_file(annotator_number):
    # Analyze one cumulative workbook per annotator. Override here if needed.
    manual_file = None
    if manual_file:
        p = LABEL_DIR / manual_file
        if not p.exists():
            raise FileNotFoundError(f"Manual annotator file does not exist: {p}")
        return p

    complete_candidates = [
        LABEL_DIR / f"human_validation_annotator{annotator_number}_500_complete.xlsm",
        LABEL_DIR / f"human_validation_annotator{annotator_number}_500_complete.xlsx",
    ]
    for p in complete_candidates:
        if p.exists():
            return p

    candidates = [
        p for p in LABEL_DIR.glob(f"human_validation_annotator{annotator_number}_500*.xls*")
        if not p.name.startswith("~$")
    ]
    if not candidates:
        raise FileNotFoundError(f"No annotator {annotator_number} Excel workbook found in {LABEL_DIR}")
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]


def read_annotation_file(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        xl = pd.ExcelFile(path)
        sheet = "Annotation" if "Annotation" in xl.sheet_names else xl.sheet_names[0]
        df = pd.read_excel(path, sheet_name=sheet)
    df.columns = [str(c).strip() for c in df.columns]
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string").str.strip()
    for col in ANNOT_COLS:
        if col not in df.columns:
            raise ValueError(f"{path.name} is missing required column: {col}")
        df[col] = canon_series(df[col])
    df["sample_id"] = df["sample_id"].astype("string").str.strip()
    df = df
    return df


def completed_mask(df):
    return df[ANNOT_COLS].notna().all(axis=1)


def completed_range(df):
    ids = df.loc[completed_mask(df), "sample_id"].astype(str).tolist()
    if not ids:
        return "none"
    return f"{ids[0]} to {ids[-1]}"


def validity_check(df, annotator_name):
    rows = []
    allowed = {
        "human_overall_sentiment": set(SENT5),
        "human_efficacy": set(ASPECT_VALUES),
        "human_safety": set(ASPECT_VALUES),
        "human_burden": set(ASPECT_VALUES),
        "human_cost": set(ASPECT_VALUES),
        "human_uncertain": set(YES_NO),
    }
    complete = completed_mask(df)
    for col, valid in allowed.items():
        values = set(df.loc[complete, col].dropna().astype(str))
        bad = sorted(values - valid)
        rows.append({"annotator": annotator_name, "field": col, "invalid_values": "; ".join(bad)})
    return pd.DataFrame(rows)


def source_integrity_check(df, key, annotator_name):
    source = df[SOURCE_COLS].merge(
        key[SOURCE_COLS],
        on="sample_id",
        how="left",
        suffixes=("_human", "_source"),
        indicator=True,
    )
    rows = [{"annotator": annotator_name, "field": "sample_id_found_in_key", "mismatch_type": "raw", "mismatches": int((source["_merge"] != "both").sum())}]
    for col in SOURCE_COLS[1:]:
        a = source[f"{col}_human"].astype("string").fillna("")
        b = source[f"{col}_source"].astype("string").fillna("")
        rows.append({"annotator": annotator_name, "field": col, "mismatch_type": "raw", "mismatches": int((a != b).sum())})
        if col == "review":
            rows.append({"annotator": annotator_name, "field": col, "mismatch_type": "whitespace_normalized", "mismatches": int((a.map(norm_ws) != b.map(norm_ws)).sum())})
    return pd.DataFrame(rows)


def pct_agreement(a, b):
    a = pd.Series(a).astype("string")
    b = pd.Series(b).astype("string")
    mask = a.notna() & b.notna()
    if mask.sum() == 0:
        return np.nan
    return float((a[mask] == b[mask]).mean())


def nominal_alpha_two_raters(a, b):
    # Nominal Krippendorff alpha for two complete raters.
    a = pd.Series(a).astype("string")
    b = pd.Series(b).astype("string")
    mask = a.notna() & b.notna()
    a = a[mask].astype(str).to_numpy()
    b = b[mask].astype(str).to_numpy()
    n = len(a)
    if n == 0:
        return np.nan
    do = np.mean(a != b)
    pooled = np.concatenate([a, b])
    counts = Counter(pooled)
    total = len(pooled)
    if total <= 1:
        return np.nan
    de = 1.0 - sum((c / total) ** 2 for c in counts.values())
    if math.isclose(de, 0.0):
        return np.nan
    return 1.0 - do / de


def safe_kappa(a, b, labels, weights=None):
    a = pd.Series(a).astype("string")
    b = pd.Series(b).astype("string")
    mask = a.notna() & b.notna() & a.isin(labels) & b.isin(labels)
    a = a[mask].astype(str)
    b = b[mask].astype(str)
    n = len(a)
    if n == 0:
        return np.nan
    k = len(labels)
    label_to_idx = {label: i for i, label in enumerate(labels)}
    conf = np.zeros((k, k), dtype=float)
    for x, y in zip(a, b):
        conf[label_to_idx[x], label_to_idx[y]] += 1.0
    row = conf.sum(axis=1)
    col = conf.sum(axis=0)
    expected = np.outer(row, col) / n
    if weights is None:
        po = np.trace(conf) / n
        pe = np.trace(expected) / n
        if math.isclose(1.0 - pe, 0.0):
            return np.nan
        return float((po - pe) / (1.0 - pe))
    if k <= 1:
        return np.nan
    idx = np.arange(k)
    dist = np.abs(idx[:, None] - idx[None, :])
    if weights == "linear":
        w = dist / (k - 1)
    elif weights == "quadratic":
        w = (dist ** 2) / ((k - 1) ** 2)
    else:
        raise ValueError(f"Unknown weights: {weights}")
    observed_disagreement = float((w * conf).sum())
    expected_disagreement = float((w * expected).sum())
    if math.isclose(expected_disagreement, 0.0):
        return np.nan
    return float(1.0 - observed_disagreement / expected_disagreement)


def compare_series(left, right, labels, ordinal=False, n_boot=0, seed=2026):
    left = pd.Series(left).astype("string")
    right = pd.Series(right).astype("string")
    mask = left.notna() & right.notna()
    l = left[mask].astype(str).reset_index(drop=True)
    r = right[mask].astype(str).reset_index(drop=True)
    n = len(l)
    out = {
        "n": n,
        "agreement": pct_agreement(l, r),
        "cohen_kappa": safe_kappa(l, r, labels=labels, weights=None),
        "krippendorff_alpha_nominal": nominal_alpha_two_raters(l, r),
        "linear_weighted_kappa": safe_kappa(l, r, labels=labels, weights="linear") if ordinal else np.nan,
        "quadratic_weighted_kappa": safe_kappa(l, r, labels=labels, weights="quadratic") if ordinal else np.nan,
    }
    if n_boot and n > 1:
        rng = np.random.default_rng(seed)
        boot_agree = []
        boot_kappa = []
        idx = np.arange(n)
        for _ in range(n_boot):
            sample_idx = rng.choice(idx, size=n, replace=True)
            boot_agree.append(pct_agreement(l.iloc[sample_idx], r.iloc[sample_idx]))
            boot_kappa.append(safe_kappa(l.iloc[sample_idx], r.iloc[sample_idx], labels=labels, weights=None))
        out["agreement_ci_low"] = float(np.nanpercentile(boot_agree, 2.5))
        out["agreement_ci_high"] = float(np.nanpercentile(boot_agree, 97.5))
        out["cohen_kappa_ci_low"] = float(np.nanpercentile(boot_kappa, 2.5))
        out["cohen_kappa_ci_high"] = float(np.nanpercentile(boot_kappa, 97.5))
    else:
        out["agreement_ci_low"] = np.nan
        out["agreement_ci_high"] = np.nan
        out["cohen_kappa_ci_low"] = np.nan
        out["cohen_kappa_ci_high"] = np.nan
    return out


def compare_frames(left_df, right_df, left_col, right_col, comparison_group, target, labels, ordinal=False, n_boot=1000):
    left_small = left_df[["sample_id", left_col]].copy().rename(columns={left_col: "left_label"})
    right_small = right_df[["sample_id", right_col]].copy().rename(columns={right_col: "right_label"})
    merged = left_small.merge(right_small, on="sample_id", how="inner")
    metric = compare_series(merged["left_label"], merged["right_label"], labels=labels, ordinal=ordinal, n_boot=n_boot)
    metric.update({
        "comparison_group": comparison_group,
        "target": target,
        "left_col": left_col,
        "right_col": right_col,
    })
    metric = {"comparison_group": metric.pop("comparison_group"), "target": metric.pop("target"), "left_col": metric.pop("left_col"), "right_col": metric.pop("right_col"), **metric}

    conf = pd.crosstab(
        pd.Categorical(merged["left_label"], categories=labels),
        pd.Categorical(merged["right_label"], categories=labels),
        dropna=False,
    )
    conf.index.name = "left_label"
    conf.columns.name = "right_label"
    conf_long = conf.reset_index().melt(id_vars="left_label", var_name="right_label", value_name="n")
    conf_long.insert(0, "comparison_group", comparison_group)
    conf_long.insert(1, "target", target)

    disagree = merged[merged["left_label"].astype("string") != merged["right_label"].astype("string")].copy()
    disagree.insert(0, "comparison_group", comparison_group)
    disagree.insert(1, "target", target)
    return metric, conf_long, disagree[["comparison_group", "target", "sample_id", "left_label", "right_label"]]


In [ ]:
# Load key and one cumulative annotator file per annotator.
if not KEY_PATH.exists():
    raise FileNotFoundError(f"Missing key file: {KEY_PATH}")

key = pd.read_csv(KEY_PATH)
# The LLM labels come from the frozen annotation release, matched to the key by row position and review text.
if not RELEASE_PATH.exists():
    raise FileNotFoundError(f"Missing annotation release: {RELEASE_PATH}")
_rel = pd.read_csv(RELEASE_PATH, usecols=["uniqueID", "review", "ai_sent5_hard", "ai_efficacy", "ai_safety", "ai_burden", "ai_cost"])
_rows = _rel.iloc[key["source_row_id"].values].reset_index(drop=True)
import html as _html
def _norm(s):
    s = _html.unescape(str(s)).strip()
    if len(s) > 1 and s[0] == s[-1] == '"':
        s = s[1:-1].strip()
    return " ".join(s.split())
_mismatch = int((_rows["review"].map(_norm) != key["review"].map(_norm)).sum())
if _mismatch:
    raise ValueError(f"{_mismatch} key rows do not match the release text at their source_row_id; the key and the release are out of step")
key["uniqueID"] = _rows["uniqueID"].values
for _c in ["ai_sent5_hard", "ai_efficacy", "ai_safety", "ai_burden", "ai_cost"]:
    key[_c] = _rows[_c].values
key = key[[c for c in ["sample_id", "source_row_id", "uniqueID", "drugName", "ingredient_norm", "condition", "condition_norm", "review",
                    "ai_sent5_hard", "ai_efficacy", "ai_safety", "ai_burden", "ai_cost"] if c in key.columns]]
print("LLM labels taken from the annotation release for", len(key), "reviews; text check passed")
key["sample_id"] = key["sample_id"].astype("string").str.strip()
for col in ["ai_sent5_hard", "ai_efficacy", "ai_safety", "ai_burden", "ai_cost"]:
    key[col] = canon_series(key[col])

A1_FILE = find_cumulative_annotator_file(1)
A2_FILE = find_cumulative_annotator_file(2)

a1 = read_annotation_file(A1_FILE)
a2 = read_annotation_file(A2_FILE)

print("Annotator 1 file:", A1_FILE.name)
print("Annotator 2 file:", A2_FILE.name)
print("Annotator 1 completed:", int(completed_mask(a1).sum()), completed_range(a1))
print("Annotator 2 completed:", int(completed_mask(a2).sum()), completed_range(a2))

a1_completed = a1.loc[completed_mask(a1)].copy()
a2_completed = a2.loc[completed_mask(a2)].copy()

qc_summary = pd.DataFrame([
    {
        "annotator": "annotator1",
        "file": A1_FILE.name,
        "rows": len(a1),
        "completed_rows": int(len(a1_completed)),
        "completed_range": completed_range(a1),
        "human_uncertain_yes": int((a1_completed["human_uncertain"] == "YES").sum()),
    },
    {
        "annotator": "annotator2",
        "file": A2_FILE.name,
        "rows": len(a2),
        "completed_rows": int(len(a2_completed)),
        "completed_range": completed_range(a2),
        "human_uncertain_yes": int((a2_completed["human_uncertain"] == "YES").sum()),
    },
])

validity = pd.concat([
    validity_check(a1, "annotator1"),
    validity_check(a2, "annotator2"),
], ignore_index=True)

source_integrity = pd.concat([
    source_integrity_check(a1, key, "annotator1"),
    source_integrity_check(a2, key, "annotator2"),
], ignore_index=True)

display(qc_summary)
display(validity)
display(source_integrity)


In [ ]:
# Build agreement comparisons.
ai_df = key[["sample_id", "ai_sent5_hard", "ai_efficacy", "ai_safety", "ai_burden", "ai_cost"]].copy()

metric_rows = []
confusion_tables = []
disagreement_tables = []

def add_comparison(left_df, right_df, left_col, right_col, comparison_group, target, labels, ordinal=False):
    metric, conf, disagree = compare_frames(
        left_df=left_df,
        right_df=right_df,
        left_col=left_col,
        right_col=right_col,
        comparison_group=comparison_group,
        target=target,
        labels=labels,
        ordinal=ordinal,
        n_boot=1000,
    )
    metric_rows.append(metric)
    confusion_tables.append(conf)
    disagreement_tables.append(disagree)


for ann_name, ann_df in [("annotator1", a1_completed), ("annotator2", a2_completed)]:
    group = f"4omini_vs_{ann_name}"
    add_comparison(ai_df, ann_df, "ai_sent5_hard", "human_overall_sentiment", group, "overall_holistic_5class", SENT5, ordinal=True)
    for aspect in ASPECTS:
        add_comparison(ai_df, ann_df, f"ai_{aspect}", f"human_{aspect}", group, aspect, ASPECT_VALUES, ordinal=False)

group = "annotator1_vs_annotator2"
add_comparison(a1_completed, a2_completed, "human_overall_sentiment", "human_overall_sentiment", group, "overall_5class", SENT5, ordinal=True)
for aspect in ASPECTS:
    add_comparison(a1_completed, a2_completed, f"human_{aspect}", f"human_{aspect}", group, aspect, ASPECT_VALUES, ordinal=False)

agreement_summary = pd.DataFrame(metric_rows)
confusion_long = pd.concat(confusion_tables, ignore_index=True)
disagreement_long = pd.concat(disagreement_tables, ignore_index=True)

# Add context to disagreement rows for manual review.
context_cols = ["sample_id", "ingredient_norm", "condition_norm", "review"]
disagreement_with_context = disagreement_long.merge(key[context_cols], on="sample_id", how="left")
disagreement_with_context["review_excerpt"] = disagreement_with_context["review"].astype(str).str.replace(r"\s+", " ", regex=True).str.slice(0, 500)
disagreement_with_context = disagreement_with_context.drop(columns=["review"])

display_cols = [
    "comparison_group", "target", "n", "agreement", "agreement_ci_low", "agreement_ci_high",
    "cohen_kappa", "cohen_kappa_ci_low", "cohen_kappa_ci_high",
    "krippendorff_alpha_nominal", "linear_weighted_kappa", "quadratic_weighted_kappa",
]
display(agreement_summary[display_cols].sort_values(["comparison_group", "target"]))


In [ ]:
# Label distributions.
dist_rows = []

def add_distribution(df, source_name, columns):
    for col in columns:
        if col not in df.columns:
            continue
        counts = df[col].value_counts(dropna=False).sort_index()
        total = counts.sum()
        for value, n in counts.items():
            dist_rows.append({
                "source": source_name,
                "field": col,
                "value": str(value),
                "n": int(n),
                "pct": float(n / total) if total else np.nan,
            })

add_distribution(ai_df, "4omini", ["ai_sent5_hard", "ai_efficacy", "ai_safety", "ai_burden", "ai_cost"])
add_distribution(a1_completed, "annotator1", ["human_overall_sentiment", "human_efficacy", "human_safety", "human_burden", "human_cost", "human_uncertain"])
add_distribution(a2_completed, "annotator2", ["human_overall_sentiment", "human_efficacy", "human_safety", "human_burden", "human_cost", "human_uncertain"])

label_distributions = pd.DataFrame(dist_rows)
display(label_distributions.head(40))


In [ ]:
# Save all outputs.
qc_summary.to_csv(OUT_DIR / "human_validation_qc_summary.csv", index=False)
validity.to_csv(OUT_DIR / "human_validation_invalid_values.csv", index=False)
source_integrity.to_csv(OUT_DIR / "human_validation_source_integrity.csv", index=False)
agreement_summary.to_csv(OUT_DIR / "human_validation_agreement_summary.csv", index=False)
confusion_long.to_csv(OUT_DIR / "human_validation_confusion_matrices_long.csv", index=False)
disagreement_with_context.to_csv(OUT_DIR / "human_validation_disagreements_with_context.csv", index=False)
label_distributions.to_csv(OUT_DIR / "human_validation_label_distributions.csv", index=False)

# A manuscript-friendly compact table.
compact = agreement_summary[[
    "comparison_group", "target", "n", "agreement", "cohen_kappa",
    "krippendorff_alpha_nominal", "linear_weighted_kappa", "quadratic_weighted_kappa",
]].copy()
for col in ["agreement", "cohen_kappa", "krippendorff_alpha_nominal", "linear_weighted_kappa", "quadratic_weighted_kappa"]:
    compact[col] = compact[col].round(3)
compact.to_csv(OUT_DIR / "human_validation_agreement_summary_compact.csv", index=False)

print("Saved outputs:")
for p in sorted(OUT_DIR.glob("human_validation_*.csv")):
    print(" -", p)

display(compact.sort_values(["comparison_group", "target"]))


In [ ]:
# Optional quick visualization: agreement and kappa by comparison.
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Plotting skipped because matplotlib is unavailable: {exc}")
    plt = None

plot_df = compact.copy()
plot_df["label"] = plot_df["comparison_group"] + " | " + plot_df["target"]
plot_df = plot_df.sort_values(["comparison_group", "target"])

if plt is not None:
    fig_h = max(6, 0.35 * len(plot_df))
    fig, ax = plt.subplots(figsize=(10, fig_h))
    ax.barh(plot_df["label"], plot_df["agreement"], color="#4C78A8")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Percent agreement")
    ax.set_title("Human Validation Agreement")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "human_validation_percent_agreement.png", dpi=200)
    plt.show()

    fig, ax = plt.subplots(figsize=(10, fig_h))
    ax.barh(plot_df["label"], plot_df["cohen_kappa"], color="#F58518")
    ax.set_xlim(-0.1, 1)
    ax.set_xlabel("Cohen kappa")
    ax.set_title("Human Validation Cohen Kappa")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "human_validation_cohen_kappa.png", dpi=200)
    plt.show()
